In [ ]:
%pip install transformers accelerate torch torchvision

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


In [ ]:
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from PIL import Image
import torch

class Blip2QA:
    def __init__(self, model_name="Salesforce/blip2-flan-t5-xl", dtype=torch.float16):
        # Original code for CUDA GPU system
        # self.processor = Blip2Processor.from_pretrained(model_name)
        # self.model = Blip2ForConditionalGeneration.from_pretrained(
        #     model_name,
        #     device_map="auto",
        #     torch_dtype=dtype
        # )

        # For macOS system: Detect and set MPS device
        if torch.backends.mps.is_available():
            self.device = torch.device("mps")
            print("Using MPS (Metal Performance Shaders) acceleration")
        else:
            self.device = torch.device("cpu")
            print("Using CPU")
        
        # Optimized processor loading (eliminates use_fast warning)
        self.processor = Blip2Processor.from_pretrained(
            model_name,
            use_fast=True  # Explicitly specify using fast processor
        )
        
        self.model = Blip2ForConditionalGeneration.from_pretrained(
            model_name,
            dtype=dtype,  # Use dtype instead of torch_dtype
            low_cpu_mem_usage=True,
            device_map={"": self.device}
        )
        

    def ask_question(self, image_path, question, caption=None, max_new_tokens=30):
        image = Image.open(image_path).convert("RGB")
        prompt = f"Question: {question}"
    
        # Original code for CUDA GPU system
        # inputs = self.processor(images=image, text=prompt, return_tensors="pt").to(self.model.device, torch.float16)
        
        # New code: Optimized device allocation
        inputs = self.processor(images=image, text=prompt, return_tensors="pt")
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            generated_ids = self.model.generate(**inputs, max_new_tokens=max_new_tokens)
            answer = self.processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
        
        return answer

qa_model = Blip2QA()
# image_path = "/media/gp921526/Thanet/Work/Multi_Agent_Cartoon/dataset/simpsons/val_images/S33/S33E13-04701.jpg"
# image_path = "/Users/wt/PythonProjects/Multi_Agent_Cartoon/dataset/simpsons/val_images/S33/S33E13-04701.jpg"
# question = "what is in the background?" 
# image_path = "/Users/wt/PythonProjects/Multi_Agent_Cartoon/dataset/simpsons/val_images/S33/S33E13-09001.jpg"
# question = "what is behind the men?" 
image_path = "/Users/wt/PythonProjects/Multi_Agent_Cartoon/dataset/simpsons/val_images/S27/S27E20-04751.jpg"
question = "what is the group of people doing?" 
answer = qa_model.ask_question(image_path, question)
print("Answer:", answer)

/Users/wt/PythonProjects/Multi_Agent_Cartoon/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using MPS (Metal Performance Shaders) acceleration


Loading checkpoint shards: 100%|██████████| 2/2 [00:25<00:00, 12.59s/it]


Answer: walking down a hallway
